### Semantic Chunking
Semantic Chunking is a document splitter that uses embedding similarity between sentences to decide chunk boundaries. It ensures that each chunk is semantically coherent and not cut off mid-thought like traditional character/token splitters.
Steps to be followed.
<ol>
<li>Document Segmentation</li>
<li>Sentence Embedding</li>
<li>Semantic Similarity Check</li>
<li>Merging of Sentences based on Cosine Similarity</li>
<li>Form the chunks based on similarity</li>
</ol>

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.chat_models import init_chat_model
from langchain_core.runnables import RunnableMap, RunnablePassthrough, RunnableLambda
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import numpy as np

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


### Custom Semantic Chunker with Threshold

In [3]:
class ThresholdSemanticChunker:
    def __init__(self, model_name="all-MiniLM-L6-v2", threshold=0.7):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold

    def split(self, text: str):
        sentences=[s.strip() for s in text.split("\n") if s.strip()]
        embeddings = self.model.encode(sentences)
        chunks = []
        current_chunk = [sentences[0]]  # Start with the first sentence

        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i]], [embeddings[i-1]])[0][0]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(". ".join(current_chunk) + ".")  # End current chunk and start a new one
                current_chunk = [sentences[i]]  

        # Add the last chunk
        if current_chunk:
            chunks.append(". ".join(current_chunk) + ".")  

        return chunks
    
    def split_documents(self, docs):
        result=[]
        for doc in docs:
            for chunk in self.split(doc.page_content):
                result.append(Document(page_content=chunk, metadata=doc.metadata))
        return result

In [4]:
# Sample text
text="""LangChain is a framework for building applications with LLMs.
LangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is lcoated in Paris.
France is a popular tourist destination."""

doc = Document(page_content=text, metadata={"source": "sample.txt"})
doc

Document(metadata={'source': 'sample.txt'}, page_content='LangChain is a framework for building applications with LLMs.\nLangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is lcoated in Paris.\nFrance is a popular tourist destination.')

In [5]:
# Chunking
chunker=ThresholdSemanticChunker(threshold=0.7)
chunks=chunker.split_documents([doc])
chunks

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Document(metadata={'source': 'sample.txt'}, page_content='LangChain is a framework for building applications with LLMs.. LangChain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone..'),
 Document(metadata={'source': 'sample.txt'}, page_content='You can create chains, agents, memory, and retrievers..'),
 Document(metadata={'source': 'sample.txt'}, page_content='The Eiffel Tower is lcoated in Paris..'),
 Document(metadata={'source': 'sample.txt'}, page_content='France is a popular tourist destination..')]

In [6]:
# VectorStore
embeddings=OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1536)
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore

In [ ]:
retriever = vectorstore.as_retriever() # Retriever is getting it from Semantic Chunking
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002ABEE2123D0>, search_kwargs={})

In [8]:
# Prompt Template

template = """Answer the question based on the following context:

{context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n\n{context}\n\nQuestion: {question}\n')

In [9]:
# LLM model
chat_model = init_chat_model(
    model="gpt-3.5-turbo",
    temperature=0.2,
    max_tokens=500
)

llm=init_chat_model("openai:gpt-3.5-turbo")
#llm=init_chat_model("groq:modelname")
llm
llm.invoke("What is AI?")

AIMessage(content='AI, or artificial intelligence, refers to the development of computer systems that can perform tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation. AI technology aims to create machines that can learn, adapt, and solve problems in a way that mimics human intelligence. This can include techniques such as machine learning, deep learning, and natural language processing.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 11, 'total_tokens': 90, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DMIpyGKqaJA1U0QyMCegZtfsyxjxU', 'service_tier': 'default', 'finish_rea

### LCEL Chain with retrieval

In [10]:
rag_chain = (RunnableMap(
        {
            "context": lambda x: retriever.invoke(x["question"]),
            "question": lambda x: x["question"]
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)
rag_chain

{
  context: RunnableLambda(...),
  question: RunnableLambda(...)
}
| PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n\n{context}\n\nQuestion: {question}\n')
| ChatOpenAI(profile={'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': False, 'structured_output': False, 'image_url_inputs': False, 'pdf_inputs': False, 'pdf_tool_message': False, 'image_tool_message': False, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002ABF00E5ED0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002ABF00E63D0>, root_client=<openai.OpenAI object at 0x000002ABF00E5AD0>, root_a

### Run Query

In [11]:
# Run Query
query = {"question": "Where is LangChain used for?"}
result = rag_chain.invoke(query)
result

'LangChain is used for building applications with Large Language Models (LLMs) and combining them with tools like OpenAI and Pinecone.'